In [1]:
%uv pip install torch triton

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 16ms
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [6]:
@triton.jit
def matpose_k(
    input_ptr, output_ptr,
    rows, cols,
    BLOCK_SIZE:tl.constexpr
):
    pid_row = tl.program_id(0)
    pid_col = tl.program_id(1)

    row_offs = (
        pid_row * BLOCK_SIZE
        + tl.arange(0, BLOCK_SIZE)
    )

    col_offs = (
        pid_col * BLOCK_SIZE
        + tl.arange(0, BLOCK_SIZE)
    )

    input_offs = (
        row_offs[:, None] * cols
        + col_offs[None, :]
    )
    input_mask = (
        (row_offs[:, None] < rows) & (col_offs[None, :] < cols)
    )

    tile = tl.load(input_ptr + input_offs, mask=input_mask, other=0.0)

    tposed_tile = tl.trans(tile)

    output_row_offs = col_offs
    output_col_offs = row_offs

    output_offs = (
        output_row_offs[:, None]*rows
        + output_col_offs[None, :]
    )

    output_mask = (
        (output_row_offs[:, None] < cols) & (output_col_offs[None, :] < rows)
    )

    tl.store(output_ptr+output_offs, tposed_tile, mask=output_mask)

In [7]:
def triton_matpose(x: torch.Tensor):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim == 2
    assert x.is_contiguous()

    rows, cols = x.shape

    output = torch.empty(
        (cols, rows),
        device=x.device,
        dtype=x.dtype
    )

    block_size = 32

    grid = (
        triton.cdiv(rows, block_size),
        triton.cdiv(cols, block_size)
    )

    matpose_k[grid](
        x, output,
        rows, cols,
        BLOCK_SIZE=block_size,
        num_warps=8
    )

    return output

In [8]:
x = torch.tensor(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ],
    device=DEVICE,
    dtype=torch.float32,
)

torch_output = x.transpose(0, 1).contiguous()
triton_output = triton_matpose(x)

print("Torch:")
print(torch_output)

print("\nTriton:")
print(triton_output)

print(
    "\nMax difference:",
    torch.max(torch.abs(torch_output - triton_output)),
)

print(
    "Results match:",
    torch.allclose(torch_output, triton_output),
)

torch.testing.assert_close(
    triton_output,
    torch_output,
)

Torch:
tensor([[1., 4.],
        [2., 5.],
        [3., 6.]], device='cuda:0')

Triton:
tensor([[1., 4.],
        [2., 5.],
        [3., 6.]], device='cuda:0')

Max difference: tensor(0., device='cuda:0')
Results match: True


In [9]:
test_shapes = [
    (1, 1),
    (2, 3),
    (3, 2),
    (31, 47),
    (32, 32),
    (33, 65),
    (1000, 1500),
]

for rows, cols in test_shapes:
    x = torch.randn(
        (rows, cols),
        device=DEVICE,
        dtype=torch.float32,
    )

    torch_output = (
        x.transpose(0, 1)
        .contiguous()
    )

    triton_output = triton_matpose(x)

    torch.testing.assert_close(
        triton_output,
        torch_output,
        rtol=0,
        atol=0,
    )

    print(
        f"{rows}x{cols} -> "
        f"{cols}x{rows}: passed"
    )

1x1 -> 1x1: passed
2x3 -> 3x2: passed
3x2 -> 2x3: passed
31x47 -> 47x31: passed
32x32 -> 32x32: passed
33x65 -> 65x33: passed
1000x1500 -> 1500x1000: passed
